# COVID-19 vs. Viral Pneumonia vs. Normal: Chest X-ray Classification

Three-class classification of chest radiographs, comparing models of increasing capacity:

| # | Model | Idea |
|---|-------|------|
| 1 | **k-NN** on raw pixels | classical baseline |
| 2 | **Custom CNN** (4 conv layers + spectral normalization), trained from scratch | what a small hand-designed network can learn |
| 3 | **ResNet-50**, ImageNet-pretrained, fine-tuned | transfer learning |

The notebook also covers **test-time augmentation (TTA)**, class-imbalance-aware metrics, and **Grad-CAM** to check *where* the networks look.

**Evaluation protocol.** The official test split is never augmented, resampled or used for model selection.
Model selection uses a stratified validation split carved out of the training data *before* any augmentation,
so augmented copies of one image can never end up on both sides of a split.

> Runs end-to-end on Google Colab (Runtime → Change runtime type → GPU), ~10 min.

## 0. Setup

In [ ]:
import os, random, copy, time, json
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import spectral_norm
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms, models
import torchvision.transforms.functional as TF

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
QUICK = os.environ.get("QUICK") == "1"          # tiny run for smoke-testing only
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)
print("device:", DEVICE, "| quick mode:", QUICK)

## 1. Data

[COVID-19 Image Dataset](https://www.kaggle.com/datasets/pranavraikokte/covid19-image-dataset) (Kaggle, Pranav Raikokte):
317 chest X-rays in three classes, pre-split into `train/` and `test/`.

In [ ]:
DATA_DIR = os.environ.get("DATA_DIR")
if DATA_DIR is None:
    import kagglehub  # preinstalled on Colab; otherwise: pip install kagglehub
    DATA_DIR = os.path.join(kagglehub.dataset_download("pranavraikokte/covid19-image-dataset"), "Covid19-dataset")
DATA_DIR = Path(DATA_DIR)

CLASSES = ["Covid", "Normal", "Viral Pneumonia"]
IMG_EXT = {".jpg", ".jpeg", ".png"}

def list_split(split):
    paths, labels = [], []
    for idx, cls in enumerate(CLASSES):
        for p in sorted((DATA_DIR / split / cls).iterdir()):
            if p.suffix.lower() in IMG_EXT:
                paths.append(p); labels.append(idx)
    return paths, np.array(labels)

train_paths_all, train_labels_all = list_split("train")
test_paths, test_labels = list_split("test")

counts = pd.DataFrame({split: Counter(lbl) for split, lbl in [("train", train_labels_all), ("test", test_labels)]})
counts.index = [CLASSES[i] for i in counts.index]
counts

In [ ]:
sizes = pd.DataFrame([(CLASSES[l], *Image.open(p).size) for p, l in zip(train_paths_all, train_labels_all)],
                     columns=["class", "width", "height"])
print(sizes.groupby("class")[["width", "height"]].median())

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for r, cls in enumerate(CLASSES):
    idx = np.where(train_labels_all == r)[0][:4]
    for c, i in enumerate(idx):
        axes[r, c].imshow(Image.open(train_paths_all[i]).convert("L"), cmap="gray")
        axes[r, c].set_title(cls, fontsize=9); axes[r, c].axis("off")
plt.tight_layout(); plt.savefig(FIG_DIR / "samples.png", dpi=120, bbox_inches="tight"); plt.show()

Observations:
* **Class imbalance**: COVID is over-represented in both splits. Instead of writing synthetic images to disk,
  training uses a **weighted sampler** (each class drawn equally often) with **on-the-fly augmentation**,
  and results are reported with **balanced accuracy** and **macro-F1**, which are insensitive to class proportions.
* **Image sizes differ strongly by class** (median resolution is a giveaway of the source). This is a known shortcut risk
  for this dataset; see *Limitations* at the end.

### Train / validation split and data pipeline

In [ ]:
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_paths_all, train_labels_all, test_size=0.2, stratify=train_labels_all, random_state=SEED)
print(f"train {len(train_paths)} | val {len(val_paths)} | test {len(test_paths)} (test is never touched)")

IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet statistics

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class XrayDataset(Dataset):
    def __init__(self, paths, labels, tf):
        self.paths, self.labels, self.tf = list(paths), np.asarray(labels), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), int(self.labels[i])

def balanced_sampler(labels, epoch_size):
    w = 1.0 / np.bincount(labels)[labels]
    return WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=epoch_size,
                                 replacement=True, generator=torch.Generator().manual_seed(SEED))

BATCH = 32
EPOCH_SIZE = 3 * 400   # samples drawn per epoch (~400 per class, as in the original course project)
if QUICK: EPOCH_SIZE = 64

train_ds = XrayDataset(train_paths, train_labels, train_tf)
val_ds   = XrayDataset(val_paths, val_labels, eval_tf)
test_ds  = XrayDataset(test_paths, test_labels, eval_tf)

g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=balanced_sampler(train_labels, EPOCH_SIZE), num_workers=2, generator=g)
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2)

### Evaluation helpers

In [ ]:
RESULTS = []

def report(name, y_true, y_pred, save_as=None):
    row = {"model": name,
           "accuracy": accuracy_score(y_true, y_pred),
           "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
           "macro_f1": f1_score(y_true, y_pred, average="macro")}
    RESULTS.append(row)
    print(f"== {name} ==")
    print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3, zero_division=0))
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=range(3)), display_labels=CLASSES).plot(
        cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(name); plt.xticks(rotation=20)
    if save_as: plt.savefig(FIG_DIR / save_as, dpi=120, bbox_inches="tight")
    plt.show()
    return row

# Test-time augmentation: average softmax over a few deterministic views of each test image.
TTA_VIEWS = [
    lambda x: x,
    lambda x: TF.hflip(x),
    lambda x: TF.rotate(x, 10),
    lambda x: TF.rotate(x, -10),
]

@torch.no_grad()
def predict_proba(model, loader, tta=False):
    model.eval(); out = []
    for x, _ in loader:
        x = x.to(DEVICE)
        views = TTA_VIEWS if tta else TTA_VIEWS[:1]
        out.append(torch.stack([F.softmax(model(v(x)), 1) for v in views]).mean(0).cpu())
    return torch.cat(out).numpy()

def fit(model, epochs, optimizer, scheduler=None):
    # Trains for a fixed number of epochs and keeps the weights with the best validation balanced accuracy.
    crit = nn.CrossEntropyLoss()
    best, best_state, hist = -1, None, []
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); loss_sum = n = 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = crit(model(x), y); loss.backward(); optimizer.step()
            loss_sum += loss.item() * len(y); n += len(y)
        if scheduler: scheduler.step()
        val_bacc = balanced_accuracy_score(val_labels, predict_proba(model, val_loader).argmax(1))
        hist.append({"epoch": ep, "train_loss": loss_sum / n, "val_bal_acc": val_bacc})
        print(f"epoch {ep:2d} | train loss {loss_sum/n:.4f} | val balanced acc {val_bacc:.3f} | {time.time()-t0:.0f}s")
        if val_bacc > best:
            best, best_state = val_bacc, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return pd.DataFrame(hist)

## 2. Baseline: k-NN on raw pixels

Images are downscaled to 64×64 grayscale and flattened (4,096 features). `k` is chosen on the validation split.

In [ ]:
def to_vec(paths, size=64):
    return np.stack([np.asarray(Image.open(p).convert("L").resize((size, size)), dtype=np.float32).ravel() / 255.
                     for p in paths])

Xtr, Xval, Xte = to_vec(train_paths), to_vec(val_paths), to_vec(test_paths)
val_scores = {k: balanced_accuracy_score(val_labels, KNeighborsClassifier(k).fit(Xtr, train_labels).predict(Xval))
              for k in [1, 3, 5, 7, 9]}
best_k = max(val_scores, key=val_scores.get)
print("validation balanced accuracy by k:", {k: round(v, 3) for k, v in val_scores.items()}, "→ k =", best_k)

knn = KNeighborsClassifier(best_k).fit(np.vstack([Xtr, Xval]), np.concatenate([train_labels, val_labels]))
report(f"k-NN (k={best_k})", test_labels, knn.predict(Xte), "cm_knn.png");

## 3. Custom CNN (trained from scratch)

A compact network designed by hand: four 7×7 convolutional blocks, each followed by ReLU and 2×2 max-pooling,
then three fully connected layers. **Spectral normalization** on the convolutions bounds each layer's Lipschitz constant,
which stabilised training on this very small dataset.

```
224×224×3 → conv7(6) → pool → conv7(16) → pool → conv7/2(32) → pool → conv7(32) → pool → 4×4×32
          → FC 256 → FC 84 → FC 3
```

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            spectral_norm(nn.Conv2d(3, 6, 7)),                       nn.ReLU(), nn.MaxPool2d(2),
            spectral_norm(nn.Conv2d(6, 16, 7, padding=1)),           nn.ReLU(), nn.MaxPool2d(2),
            spectral_norm(nn.Conv2d(16, 32, 7, stride=2, padding=1)), nn.ReLU(), nn.MaxPool2d(2),
            spectral_norm(nn.Conv2d(32, 32, 7, padding=1)),          nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 4 * 4, 256), nn.ReLU(),
            nn.Linear(256, 84), nn.ReLU(),
            nn.Linear(84, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

seed_everything()
cnn = SmallCNN().to(DEVICE)
print(f"parameters: {sum(p.numel() for p in cnn.parameters()):,}")

CNN_EPOCHS = 2 if QUICK else 20
opt = torch.optim.SGD(cnn.parameters(), lr=1e-3, momentum=0.9)
cnn_hist = fit(cnn, CNN_EPOCHS, opt)

In [ ]:
p_cnn = predict_proba(cnn, test_loader)
report("Custom CNN", test_labels, p_cnn.argmax(1), "cm_cnn.png")
p_cnn_tta = predict_proba(cnn, test_loader, tta=True)
report("Custom CNN + TTA", test_labels, p_cnn_tta.argmax(1));

## 4. Transfer learning: ResNet-50

ImageNet-pretrained ResNet-50 with the final layer replaced by a 3-way classifier; the whole network is fine-tuned
with Adam at a small learning rate.

In [ ]:
seed_everything()
weights = None if QUICK else models.ResNet50_Weights.IMAGENET1K_V2
resnet = models.resnet50(weights=weights)
resnet.fc = nn.Linear(resnet.fc.in_features, len(CLASSES))
resnet = resnet.to(DEVICE)

RESNET_EPOCHS = 1 if QUICK else 8
opt = torch.optim.Adam(resnet.parameters(), lr=1e-4)
resnet_hist = fit(resnet, RESNET_EPOCHS, opt)

In [ ]:
p_res = predict_proba(resnet, test_loader)
report("ResNet-50 (fine-tuned)", test_labels, p_res.argmax(1), "cm_resnet.png")
p_res_tta = predict_proba(resnet, test_loader, tta=True)
report("ResNet-50 + TTA", test_labels, p_res_tta.argmax(1));

## 5. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for name, h in [("Custom CNN", cnn_hist), ("ResNet-50", resnet_hist)]:
    axes[0].plot(h.epoch, h.train_loss, marker="o", label=name)
    axes[1].plot(h.epoch, h.val_bal_acc, marker="o", label=name)
axes[0].set(title="Training loss", xlabel="epoch"); axes[1].set(title="Validation balanced accuracy", xlabel="epoch", ylim=(0, 1.02))
for a in axes: a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG_DIR / "training_curves.png", dpi=120, bbox_inches="tight"); plt.show()

## 6. Interpretability: Grad-CAM

Grad-CAM highlights the image regions that drive the predicted class. A model that has learned radiological signs should
focus on the lung fields; attention on borders, text markers or the area outside the chest would point to shortcut learning.

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model, None, None
        def hook(module, inp, out):
            self.acts = out
            out.register_hook(lambda g: setattr(self, "grads", g))
        layer.register_forward_hook(hook)
    def __call__(self, x):
        self.model.eval(); x = x.to(DEVICE).requires_grad_(True)
        out = self.model(x); cls = out.argmax(1).item()
        self.model.zero_grad(); out[0, cls].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * self.acts).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), cls

def show_gradcam(model, layer, title, fname, per_class=2):
    cam = GradCAM(model, layer)
    idx = np.concatenate([np.where(test_labels == c)[0][:per_class] for c in range(3)])
    fig, axes = plt.subplots(2, len(idx), figsize=(2.6 * len(idx), 5.4))
    for j, i in enumerate(idx):
        x, y = test_ds[i]
        heat, pred = cam(x.unsqueeze(0))
        img = np.asarray(Image.open(test_paths[i]).convert("L").resize((IMG_SIZE, IMG_SIZE)))
        axes[0, j].imshow(img, cmap="gray"); axes[0, j].set_title(f"true: {CLASSES[y]}", fontsize=8)
        axes[1, j].imshow(img, cmap="gray"); axes[1, j].imshow(heat, cmap="jet", alpha=0.45)
        axes[1, j].set_title(f"pred: {CLASSES[pred]}", fontsize=8, color="green" if pred == y else "red")
        for a in axes[:, j]: a.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.savefig(FIG_DIR / fname, dpi=120, bbox_inches="tight"); plt.show()

show_gradcam(resnet, resnet.layer4, "Grad-CAM: ResNet-50 (last residual stage)", "gradcam_resnet.png")
show_gradcam(cnn, cnn.features[9], "Grad-CAM: custom CNN (last conv layer)", "gradcam_cnn.png")

## 7. Results

In [ ]:
results = pd.DataFrame(RESULTS).set_index("model").round(3)
results.to_csv("results.csv")
print(f"Test set: {len(test_labels)} untouched images ({dict(zip(CLASSES, np.bincount(test_labels).tolist()))})")
results

## 8. Limitations

* **Tiny test set.** 66 test images: one misclassification moves accuracy by ~1.5 points, so differences of a few points
  between models are within noise.
* **Shortcut risk.** In this dataset the classes come from different public sources (resolution, contrast, borders
  and text markers differ systematically by class). High scores may partly reflect *where an image came from* rather than
  pathology. Grad-CAM helps to spot this, but proper validation would require an external dataset from a different hospital.
* **Horizontal flips** are anatomically unrealistic for chest X-rays (the heart is on the left); they were kept as a
  light regulariser, but a stricter pipeline would use only small rotations, crops and intensity changes.
* Research/education project only; not a diagnostic tool.